In [3]:
from sympy import *
from IPython.display import display
import numpy as np

# ═══════════════════════════════════════════════════════════════════
#  PARAMETERS — edit here
# ═══════════════════════════════════════════════════════════════════
M = 1   # number of Floquet harmonics to keep

# ═══════════════════════════════════════════════════════════════════
#  SYMBOLS
# ═══════════════════════════════════════════════════════════════════
t, theta, omega_p = symbols('t theta omega_p', real=True)
k, n              = symbols('k n', integer=True)

# omega[q] = carrier frequency of mode q
omega = IndexedBase('omega')

# DC coefficients
Zs0 = symbols('Zs0', complex=True)
Yg0 = symbols('Yg0', complex=True)

# Zs^(m) and Yg^(m) as Functions of frequency:
#   Zs_m[mi-1](omega[q])  =  Z_s^(m) evaluated at omega_q
Zs_m = [Function(f'Zs{m}') for m in range(1, M+1)]   # Zs1, Zs2, ...
Yg_m = [Function(f'Yg{m}') for m in range(1, M+1)]

# State amplitudes: V[k+j, n],  Ic[k+j, n]
V  = IndexedBase('V')
Ic = IndexedBase('I')

# ═══════════════════════════════════════════════════════════════════
#  HARMONIC PROJECTION — arm by arm (preserves frequency arguments)
#
#  Zs(t) = Zs0
#         + sum_m  Zs_m(omega_k) * e^{+i m (omega_p t + theta)}   [+ arm]
#         + sum_m  Zs_m(omega_k) * e^{-i m (omega_p t + theta)}   [- arm]
#
#  When the + arm of Zs_m pairs with Bloch component I[k+j', n]*e^{i j' omega_p t}
#  the result lands at harmonic j' + m = j_val  =>  j' = j_val - m
#  so the correct frequency argument is omega[k + j_val - m].
#
#  When the - arm pairs with I[k+j', n]*e^{i j' omega_p t}
#  the result lands at harmonic j' - m = j_val  =>  j' = j_val + m
#  so the correct frequency argument is omega[k + j_val + m].
# ═══════════════════════════════════════════════════════════════════

def V_next_coeff(j_val):
    """
    Coefficient of e^{i j_val omega_p t} in V_{k,n+1}(t)
    = V_{k,n}(t) - Zs(t)*I_{k,n}(t),
    with correct omega arguments on Zs_m.
    Returns a SymPy expression in V[k+j, n] and Ic[k+j, n].
    """
    # Bloch passthrough
    out = V[j_val, n]

    # DC arm: -Zs0 * I[k+j_val, n]
    out -= Zs0 * Ic[j_val, n]

    # Floquet arms
    for mi, Zm in enumerate(Zs_m, start=1):
        # + arm: e^{+i mi omega_p t} pairs with I at harmonic j_val - mi
        if abs(j_val - mi) <= M:
            out -= Zm(omega[j_val - mi]) * exp( I*mi*theta) * Ic[j_val - mi, n]
        # - arm: e^{-i mi omega_p t} pairs with I at harmonic j_val + mi
        if abs(j_val + mi) <= M:
            out -= Zm(omega[j_val + mi]) * exp(-I*mi*theta) * Ic[j_val + mi, n]

    return expand(out)


def I_next_coeff(j_val):
    """
    Coefficient of e^{i j_val omega_p t} in I_{k,n+1}(t)
    = I_{k,n}(t) - Yg(t)*V_{k,n+1}(t),
    with correct omega arguments on Yg_m and Zs_m (via V_next_coeff).
    """
    # Bloch passthrough
    out = Ic[j_val, n]

    # DC arm of Yg: -Yg0 * V_next at harmonic j_val
    out -= Yg0 * V_next_coeff(j_val)

    # Floquet arms of Yg applied to V_next
    for mi, Ym in enumerate(Yg_m, start=1):
        # + arm: e^{+i mi omega_p t} pairs with V_next at harmonic j_val - mi
        if abs(j_val - mi) <= M:
            out -= Ym(omega[j_val - mi]) * exp( I*mi*theta) * V_next_coeff(j_val - mi)
        # - arm: e^{-i mi omega_p t} pairs with V_next at harmonic j_val + mi
        if abs(j_val + mi) <= M:
            out -= Ym(omega[j_val + mi]) * exp(-I*mi*theta) * V_next_coeff(j_val + mi)

    return expand(out)


# ═══════════════════════════════════════════════════════════════════
#  COMPUTE AND DISPLAY ALL SYMBOLIC EQUATIONS
# ═══════════════════════════════════════════════════════════════════

js      = list(range(-M, M+1))
results = {}   # results[j_val] = (V_expr, I_expr)

print("Symbolic coupled-mode equations:\n")
for j_val in js:
    cv = V_next_coeff(j_val)
    ci = I_next_coeff(j_val)
    results[j_val] = (cv, ci)
    display(Eq(V[j_val, n+1],  cv))
    display(Eq(Ic[j_val, n+1], ci))

# ═══════════════════════════════════════════════════════════════════
#  NUMERICAL SUBSTITUTION
#
#  Zs_val(m, q) and Yg_val(m, q) are functions of harmonic index m
#  and mode index q — replace with your own data / analytic formulas.
# ═══════════════════════════════════════════════════════════════════

# --- Edit numerical values here -----------------------------------
theta_val   = np.pi / 4
omega_p_val = 1.0
Zs0_val     = 1.0 + 0.0j
Yg0_val     = 0.5 + 0.0j

def Zs_val(m, q):
    """Z_s^(m) evaluated at mode index q (omega_q = q * omega_p)."""
    return {1: 0.1 + 0.05j, 2: 0.02 + 0.01j}.get(m, 0+0j)

def Yg_val(m, q):
    """Y_g^(m) evaluated at mode index q."""
    return {1: 0.05 + 0.02j, 2: 0.01 + 0.005j}.get(m, 0+0j)

def omega_val(q):
    """Carrier frequency of mode q."""
    return q * omega_p_val
# ------------------------------------------------------------------

def build_numeric_subs(k_val=0):
    """
    Build a SymPy substitution dict for a given centre mode index k_val.
    Covers all mode indices q that can appear (k_val ± 2M).
    """
    subs = {
        Zs0:    Zs0_val,
        Yg0:    Yg0_val,
        theta:  theta_val,
        omega_p: omega_p_val,
    }
    for dq in range(-2*M, 2*M+1):
        q = k_val + dq
        subs[omega[q]] = omega_val(q)
        for mi, Zm in enumerate(Zs_m, start=1):
            subs[Zm(omega[q])] = Zs_val(mi, q)
        for mi, Ym in enumerate(Yg_m, start=1):
            subs[Ym(omega[q])] = Yg_val(mi, q)
    return subs


def build_transfer_matrix(k_val=0):
    """
    Returns a (dim x dim) numpy matrix T such that
        state_{n+1} = T @ state_n
    where state = [V[k-M,n], I[k-M,n], ..., V[k+M,n], I[k+M,n]].
    """
    subs = build_numeric_subs(k_val)

    # State vector: interleaved V, I for each sideband j
    state_syms = []
    for j_val in js:
        state_syms += [V[k + j_val, n], Ic[k + j_val, n]]

    dim = len(state_syms)
    T   = np.zeros((dim, dim), dtype=complex)

    for row_idx, j_val in enumerate(js):
        cv, ci = results[j_val]
        for col_idx, s in enumerate(state_syms):
            T[2*row_idx,   col_idx] = complex(cv.coeff(s).subs(subs))
            T[2*row_idx+1, col_idx] = complex(ci.coeff(s).subs(subs))
    return T


# ═══════════════════════════════════════════════════════════════════
#  PROPAGATE N STEPS
# ═══════════════════════════════════════════════════════════════════

T_num   = build_transfer_matrix(k_val=0)
dim     = T_num.shape[0]
N_steps = 100

# Initial state: drive central mode V[k, n=0] = 1, all others = 0
state           = np.zeros(dim, dtype=complex)
state[2*M]      = 1.0   # V[k+0, n] is at index 2*M in the interleaved layout

history = np.zeros((N_steps+1, dim), dtype=complex)
history[0] = state
for step in range(N_steps):
    state          = T_num @ state
    history[step+1] = state

# history[:, 2*j]   = V[k+j, n]  for sideband index j (j=0 at column 2*M)
# history[:, 2*j+1] = I[k+j, n]

print(f"\nNumerical transfer matrix T  [shape {T_num.shape}]")
print("Real part:\n", np.round(T_num.real, 5))
print("Imag part:\n", np.round(T_num.imag, 5))

print(f"\nCentral mode V[k, n], first 10 steps:")
print(np.round(history[:10, 2*M], 6))

Symbolic coupled-mode equations:



Eq(V[-1, n + 1], -Zs0*I[-1, n] - Zs1(omega[0])*exp(-I*theta)*I[0, n] + V[-1, n])

Eq(I[-1, n + 1], Yg0*Zs0*I[-1, n] + Yg0*Zs1(omega[0])*exp(-I*theta)*I[0, n] - Yg0*V[-1, n] + Zs0*Yg1(omega[0])*exp(-I*theta)*I[0, n] + Yg1(omega[0])*Zs1(omega[-1])*I[-1, n] + Yg1(omega[0])*Zs1(omega[1])*exp(-2*I*theta)*I[1, n] - Yg1(omega[0])*exp(-I*theta)*V[0, n] + I[-1, n])

Eq(V[0, n + 1], -Zs0*I[0, n] - Zs1(omega[-1])*exp(I*theta)*I[-1, n] - Zs1(omega[1])*exp(-I*theta)*I[1, n] + V[0, n])

Eq(I[0, n + 1], Yg0*Zs0*I[0, n] + Yg0*Zs1(omega[-1])*exp(I*theta)*I[-1, n] + Yg0*Zs1(omega[1])*exp(-I*theta)*I[1, n] - Yg0*V[0, n] + Zs0*Yg1(omega[-1])*exp(I*theta)*I[-1, n] + Zs0*Yg1(omega[1])*exp(-I*theta)*I[1, n] + Yg1(omega[-1])*Zs1(omega[0])*I[0, n] - Yg1(omega[-1])*exp(I*theta)*V[-1, n] + Yg1(omega[1])*Zs1(omega[0])*I[0, n] - Yg1(omega[1])*exp(-I*theta)*V[1, n] + I[0, n])

Eq(V[1, n + 1], -Zs0*I[1, n] - Zs1(omega[0])*exp(I*theta)*I[0, n] + V[1, n])

Eq(I[1, n + 1], Yg0*Zs0*I[1, n] + Yg0*Zs1(omega[0])*exp(I*theta)*I[0, n] - Yg0*V[1, n] + Zs0*Yg1(omega[0])*exp(I*theta)*I[0, n] + Yg1(omega[0])*Zs1(omega[-1])*exp(2*I*theta)*I[-1, n] + Yg1(omega[0])*Zs1(omega[1])*I[1, n] - Yg1(omega[0])*exp(I*theta)*V[0, n] + I[1, n])


Numerical transfer matrix T  [shape (6, 6)]
Real part:
 [[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]
Imag part:
 [[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]

Central mode V[k, n], first 10 steps:
[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
